# fganalysis Agentic Workflow Demo

This notebook demonstrates how to use Google Gemini (via Vertex AI or AI Studio) with the `fganalysis` MCP server tools to perform an agentic analysis.

## Prerequisites
1. `fganalysis` R package installed.
2. `fganalysis-mcp` python package installed (`pip install -e .`).
3. `google-generativeai` installed.
4. `VERTEX_API_KEY` set in environment or `.env`.

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
from fganalysis_mcp.server import (
    run_drug_response_analysis,
    run_blup_analysis,
    get_lab_data_summary,
    get_drug_purchases,
    plot_lab_distribution
)

# Load environment variables
load_dotenv()
api_key = os.getenv("VERTEX_API_KEY")
genai.configure(api_key=api_key)

## Define Tools for Gemini
We map the MCP server functions to a format Gemini understands.

In [ ]:
tools = [
    run_drug_response_analysis,
    run_blup_analysis,
    get_lab_data_summary,
    get_drug_purchases,
    plot_lab_distribution
]

# Initialize the model with tools
model = genai.GenerativeModel(
    model_name='gemini-1.5-pro-latest',
    tools=tools
)

chat = model.start_chat(enable_automatic_function_calling=True)

## Run Agentic Analysis
We ask the agent to perform a complex task. It will automatically call the necessary tools.

In [ ]:
query = """
I want to analyze the effect of Statins (ATC code A10) on LDL cholesterol (OMOP ID 3001308).
Please follow these steps:
1. Check the summary of lab data for LDL.
2. Run a drug response analysis comparing 1 year before to 1 year after.
3. Generate a distribution plot of the results.
"""

response = chat.send_message(query)
print(response.text)

## Inspect Results
The agent should have created output files. Let's verify.

In [ ]:
import glob
print("Generated Files:")
for f in glob.glob("*.pdf") + glob.glob("*.txt"):
    print(f)